### DETECTING SUSPICIOUS REVIEWS ON YELP

This project focuses on detecting suspicious and potentially fake reviews using the Yelp Open Dataset. Since the dataset does not include labels indicating whether a review is fake or genuine, we approach the problem using anomaly detection techniques.

The goal is to identify unusual patterns in review behavior, such as extreme ratings, inconsistent user behavior, and abnormal review text characteristics. By combining data preprocessing, feature engineering, and machine learning, we aim to detect reviews that deviate from normal behavior.

Problem Statement

In [ ]:
import pandas as pd

df = pd.read_csv("modeled_reviews.csv") 

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x="stars", data=df)
plt.title("Distribution of Review Ratings")
plt.xlabel("Stars")
plt.ylabel("Count")
plt.show()

The dataset shows a strong imbalance toward higher ratings, which suggests that even a small number of extreme or suspicious reviews could influence user perception.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

## SECTION 1 : Data Loading and Initial Processing

In this section, we begin by loading the Yelp dataset, which consists of separate JSON files for reviews, businesses, and users. Since the dataset is extremely large, careful handling is required to ensure efficient processing.

We first verify that all required files are accessible before proceeding. This step helps prevent runtime errors and ensures that the rest of the pipeline executes smoothly.

In [ ]:
import pandas as pd
import json
import os
import sqlite3
from pathlib import Path

### Setting Up File Paths

Since the original Yelp dataset is extremely large and not directly available in this environment, we use a preprocessed sampled dataset.

This dataset was created by extracting a subset of reviews and merging relevant user and business information. This allows us to focus on analysis and modeling without memory constraints.

In [ ]:
BASE_DIR = Path(".")  # current folder (CodeBench)

PROCESSED_DIR = BASE_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = BASE_DIR / "sampled_reviews.csv"

print("Dataset exists:", DATA_PATH.exists())

### Loading the Dataset

We load the sampled dataset, which already contains merged review, user, and business information. This dataset was created earlier to avoid loading the full raw Yelp dataset.

In [ ]:
df_merged = pd.read_csv(DATA_PATH)

print("Dataset shape:", df_merged.shape)
df_merged.head()

### Inspecting the Dataset

We examine the structure of the dataset to understand the available features. These include review-level, user-level, and business-level attributes that will be used in later stages of the project.

In [ ]:
df_merged.columns.tolist()
df_merged.describe()

### Checking Missing Values

After merging multiple datasets, some missing values are expected. These will be handled later during feature engineering.

At this stage, we simply analyze the extent of missing data.

In [ ]:
df_merged.isnull().sum().sort_values(ascending=False)

### Saving Processed Dataset

We save the dataset again to ensure it is available for later stages of the project. This also allows separation between data preparation and modeling steps.

In [ ]:
sampled_csv_path = PROCESSED_DIR / "sampled_reviews_cleaned.csv"

df_merged.to_csv(sampled_csv_path, index=False)

print("Saved to:", sampled_csv_path)

### Storing Data in SQLite

To demonstrate database concepts, we store the dataset in a SQLite database. This allows us to perform structured queries and analyze patterns efficiently.

Using SQL also reflects real-world data pipelines where large datasets are stored and queried in relational databases.

In [ ]:
db_path = PROCESSED_DIR / "yelp_reviews.db"
conn = sqlite3.connect(db_path)

df_merged.to_sql("merged_reviews", conn, if_exists="replace", index=False)

print("Database saved to:", db_path)

### SQL Analysis: User Activity

We analyze user activity to identify users who contribute a large number of reviews. High activity levels may indicate unusual or suspicious behavior.

In [ ]:
query_1 = """
SELECT user_id, COUNT(*) AS review_count
FROM merged_reviews
GROUP BY user_id
ORDER BY review_count DESC
LIMIT 10;
"""

pd.read_sql_query(query_1, conn)

The results show that a small number of users contribute a large number of reviews, while most users appear only once. This imbalance in activity may be relevant when detecting suspicious behavior.

In [ ]:
query_2 = """
SELECT name, COUNT(*) AS reviews
FROM merged_reviews
GROUP BY name
ORDER BY reviews DESC
LIMIT 10;
"""

pd.read_sql_query(query_2, conn)

The results show that certain businesses receive a significantly higher number of reviews. This is expected for popular locations, but it also highlights how uneven data distribution can affect anomaly detection.

In [ ]:
query_3 = """
SELECT name,
       AVG(stars) AS avg_rating,
       COUNT(*) AS review_count
FROM merged_reviews
GROUP BY name
HAVING COUNT(*) >= 5
ORDER BY avg_rating DESC
LIMIT 10;
"""

pd.read_sql_query(query_3, conn)

Some businesses show very high average ratings with relatively few reviews. This pattern may indicate potential bias or artificially inflated ratings, which is relevant when identifying suspicious reviews.

In [ ]:
conn.close()

## SECTION 2 : Feature Engineering and Modeling

In this section, we build on the cleaned dataset from the previous step and focus on creating meaningful features for detecting suspicious reviews. Since the dataset does not contain labels indicating whether a review is fake, we rely on feature engineering and anomaly detection techniques to identify unusual patterns.

The goal is to capture both textual and behavioral signals that may indicate suspicious activity, and then apply machine learning models to detect anomalies.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest


### Loading Processed Dataset

We load the cleaned dataset created in the previous step. This dataset already contains merged review, user, and business information, allowing us to directly proceed with feature engineering and modeling.

In [ ]:
DATA_PATH = "sampled_reviews.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

### Inspecting Dataset Structure

Before creating features, we examine the dataset to understand available columns and identify missing values. This helps guide how we handle incomplete data and which features are useful for analysis.

In [ ]:
print(df.columns.tolist())
df.isnull().sum().sort_values(ascending=False)

### Handling Missing Values

Since the dataset was created by merging multiple sources, some values are missing. Instead of removing these rows, we handle missing values using logical assumptions.

Missing user activity fields such as review count and fans are set to zero, assuming these users are either new or have no recorded activity. Missing rating values are replaced with the overall dataset average, providing a neutral estimate.

This approach allows us to preserve as much data as possible while maintaining consistency.

In [ ]:
df["user_review_count"] = df["user_review_count"].fillna(0)
df["fans"] = df["fans"].fillna(0)

df["average_stars"] = df["average_stars"].fillna(df["stars"].mean())
df["business_avg_stars"] = df["business_avg_stars"].fillna(df["stars"].mean())

### Feature Engineering

Since the dataset does not directly indicate whether a review is fake or genuine, we create new features that may help reveal suspicious behavior.

These features are designed to capture both textual patterns and user behavior, including review length, sentiment, rating inconsistencies, and activity levels.

In [ ]:
df["review_length"] = df["text"].fillna("").apply(lambda x: len(str(x).split()))

df[["text", "review_length"]].head()

Review length is an important feature because suspicious reviews often show extreme patterns. Some may be very short and lack detail, while others may be unusually long and overly descriptive. These extremes can indicate abnormal behavior.


### Sentiment Feature

We calculate sentiment to measure how positive or negative the review text is. This helps identify inconsistencies between the written review and the star rating.

For example, a highly positive rating paired with negative language, or vice versa, may indicate suspicious behavior.

In [ ]:
def get_sentiment(text):
    text = str(text).lower()
    
    positive_words = ["good", "great", "excellent", "amazing", "love", "nice"]
    negative_words = ["bad", "terrible", "awful", "worst", "hate", "poor"]
    
    score = 0
    
    for word in positive_words:
        if word in text:
            score += 1
    
    for word in negative_words:
        if word in text:
            score -= 1
    
    return score

In [ ]:
df["sentiment"] = df["text"].apply(get_sentiment)

Instead of using external NLP libraries, we implemented a simple keyword-based sentiment scoring approach. 

This method assigns positive or negative values based on the presence of certain words in the review text. While it is less sophisticated than full NLP models, it still provides a useful approximation of the emotional tone of a review.

This approach ensures that the model remains lightweight and fully executable in the current environment.

### Rating Deviation Features

We compute how much each review deviates from expected behavior by comparing it to both the business’s average rating and the user’s typical rating pattern.

Large deviations may indicate unusual or inconsistent behavior, which is a key signal in detecting suspicious reviews.

In [ ]:
df["rating_deviation_from_business"] = abs(df["stars"] - df["business_avg_stars"])
df["rating_deviation_from_user"] = abs(df["stars"] - df["average_stars"])

A review that significantly differs from a business’s average rating or a user’s typical rating behavior may indicate abnormal activity. These deviations are strong indicators of potential anomalies.

In [ ]:
df["is_extreme_rating"] = df["stars"].apply(lambda x: 1 if x in [1, 5] else 0)

Extreme ratings, such as 1-star or 5-star reviews, are more likely to be associated with biased or manipulated feedback. This feature helps capture such patterns.

In [ ]:
feature_cols = [
    "stars",
    "review_length",
    "sentiment",
    "rating_deviation_from_business",
    "rating_deviation_from_user",
    "user_review_count",
    "is_extreme_rating"
]

X = df[feature_cols].copy()

X.head()

### Exploring Clustering Methods

Before selecting a final model, we experiment with clustering techniques such as K-Means and Hierarchical Clustering to explore patterns in the data.

These methods help group similar reviews but are not specifically designed for detecting anomalies.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
df["kmeans_cluster"] = kmeans.fit_predict(X_scaled)

print(df["kmeans_cluster"].value_counts())

K-Means grouped reviews into clusters based on similarity, but it did not clearly separate suspicious reviews. Most data points remained grouped together, making it difficult to identify anomalies.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

hc = AgglomerativeClustering(n_clusters=3)
df["hierarchical_cluster"] = hc.fit_predict(X_scaled)

print(df["hierarchical_cluster"].value_counts())

Hierarchical clustering provided a more flexible grouping approach, but similar to K-Means, it did not clearly isolate anomalous reviews. This confirmed that clustering is not ideal for anomaly detection in this problem.

### Final Model: Isolation Forest

After experimenting with clustering techniques, we selected Isolation Forest as the final model. Unlike clustering, this algorithm is specifically designed for anomaly detection.

It works by isolating data points that differ significantly from the majority, making it more effective for identifying suspicious reviews.

In [ ]:
iso = IsolationForest(
    contamination=0.02,
    random_state=42,
    n_estimators=100
)

df["anomaly_flag"] = iso.fit_predict(X_scaled)
df["anomaly_score"] = iso.decision_function(X_scaled)

df["is_suspicious"] = df["anomaly_flag"].apply(lambda x: 1 if x == -1 else 0)

print("Total suspicious reviews detected:", df["is_suspicious"].sum())

The model identifies reviews that are statistically different from the majority. These reviews are labeled as suspicious, but this does not mean they are definitively fake. Instead, they represent unusual patterns that may require further investigation.

### Limitations

The model does not confirm whether a review is fake, as no ground truth labels are available. It only identifies unusual patterns.

Additionally, some flagged reviews may be genuine but uncommon, such as detailed reviews or highly active users. Therefore, results should be interpreted carefully.

In [ ]:
output_path = "modeled_reviews.csv"
df.to_csv(output_path, index=False)

print("Saved to:", output_path)

## SECTION 3: Analysis, Visualization, and Insights

In this section, we analyze the output of the anomaly detection model and visualize patterns in suspicious reviews. The goal is to better understand how suspicious reviews differ from normal ones and identify key characteristics associated with anomalous behavior.

Through visualizations and summary analysis, we derive insights into how review structure, sentiment, rating patterns, and user activity contribute to suspicious behavior.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

DATA_PATH = "modeled_reviews.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

### Review Length Analysis

We begin by analyzing the distribution of review length across suspicious and normal reviews. This helps us understand whether unusually short or long reviews are associated with anomalous behavior.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="is_suspicious", y="review_length", data=df)
plt.title("Review Length by Suspicion Level")
plt.xlabel("Is Suspicious (0 = Normal, 1 = Suspicious)")
plt.ylabel("Review Length")
plt.show()

The visualization shows that suspicious reviews tend to have higher variation in length, with many being significantly longer than typical reviews. While long reviews are not inherently suspicious, extreme lengths may indicate unusual or exaggerated behavior.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="is_suspicious", y="sentiment", data=df)
plt.title("Sentiment Distribution by Suspicion Level")
plt.xlabel("Is Suspicious")
plt.ylabel("Sentiment Score")
plt.show()

Suspicious reviews tend to exhibit lower or more neutral sentiment scores compared to normal reviews. This suggests potential inconsistencies between the emotional tone of the text and the rating given, which may indicate unusual behavior.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="is_suspicious", y="rating_deviation_from_business", data=df)
plt.title("Rating Deviation from Business Average")
plt.xlabel("Is Suspicious")
plt.ylabel("Deviation")
plt.show()

Suspicious reviews show significantly higher deviation from business average ratings. This confirms that extreme or inconsistent ratings are one of the strongest indicators of anomalous behavior.

### User Activity Analysis

We examine how user behavior differs between suspicious and normal reviews, particularly focusing on how frequently users post reviews.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="is_suspicious", y="user_review_count", data=df)
plt.title("User Activity by Suspicion Level")
plt.xlabel("Is Suspicious")
plt.ylabel("Number of Reviews by User")
plt.show()

Suspicious reviews are often associated with highly active users. This suggests that abnormal reviewing frequency may be an important signal, potentially indicating coordinated or automated behavior.

In [ ]:
plt.figure(figsize=(8,6))

ax = sns.scatterplot(
    x="review_length",
    y="rating_deviation_from_business",
    hue="is_suspicious",
    data=df,
    alpha=0.5,
    legend=False
)

counts = df.groupby(["rating_deviation_from_business", "is_suspicious"]).size().unstack(fill_value=0)

counts["total"] = counts[0] + counts[1]
counts["ratio"] = counts[1] / counts["total"]

legend_labels = []
for dev, row in counts.head(9).iterrows():
    legend_labels.append(
        f"Dev {dev:.1f} → S:{int(row[1])}/{int(row['total'])} ({row['ratio']*100:.1f}%)"
    )

for label in legend_labels:
    ax.plot([], [], ' ', label=label)

plt.legend(title="Suspicious Ratio by Deviation", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.title("Review Length vs Rating Deviation")
plt.xlabel("Review Length")
plt.ylabel("Rating Deviation")

plt.tight_layout()
plt.show()

The scatter plot highlights the interaction between review length and rating deviation. At low deviation levels, most reviews appear normal regardless of length. However, as rating deviation increases, suspicious reviews become more frequent.

Within high deviation ranges, longer reviews are more commonly flagged as suspicious. This suggests that anomalous behavior is often driven by a combination of extreme ratings and unusual review length, rather than a single feature.

In [ ]:
pd.set_option('display.max_colwidth', None)

df[df["is_suspicious"] == 1].head(3)[["stars", "text"]]

Examining flagged reviews shows that many suspicious reviews contain unusually long or structured text. However, it is important to note that long reviews are not inherently suspicious. Many genuine users provide detailed feedback.

Instead, these reviews are identified as anomalies because they deviate from typical patterns in the dataset. The model highlights such reviews as candidates for further investigation rather than definitive cases of fraud.

An important observation from this analysis is that suspicious reviews do not necessarily aim to change overall rating averages. Instead, they are more likely to influence individual user perception through extreme or emotionally impactful content.

This suggests that deceptive reviews are designed to shape decision-making rather than simply manipulate aggregate statistics.

### Final Conclusion

This project demonstrates how feature engineering and anomaly detection can be used to identify suspicious patterns in large-scale review data.

The results show that suspicious reviews are often associated with extreme rating deviations, unusual review lengths, and high user activity. These characteristics indicate that anomalous behavior is driven by a combination of structural and behavioral factors.

While the model does not definitively identify fake reviews, it provides a practical framework for highlighting unusual patterns that may require further investigation.

### Limitations

This project uses an unsupervised anomaly detection model, which means it does not rely on labeled data. As a result, the model identifies unusual patterns rather than confirmed fraudulent activity.

Some flagged reviews may be legitimate but uncommon, such as highly detailed reviews or frequent users. Additionally, evaluation is limited since true labels are not available.

Future work could incorporate labeled datasets or more advanced models to improve accuracy and validation.